# IEDA4331 Crypto Options: Black-Scholes, Binomial Trees, and Merton Jump-Diffusion

This notebook builds the final project workflow: live Deribit BTC/ETH option chains, baseline Black-Scholes and CRR binomial pricing, Merton jump-diffusion calibration, volatility-smile diagnostics, Greeks, and jump-risk hedging simulations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_deribit import DeribitClient, fetch_market_snapshot
from src.preprocessing import normalize_option_chain, filter_liquid_options, combine_currency_frames
from src.calibration import estimate_historical_merton_params, calibrate_merton_to_options, compare_model_errors
from src.binomial import convergence_table
from src.black_scholes import bs_greeks
from src.risk import simulate_jump_hedge_experiment
from src.plots import set_project_style, plot_return_jumps, plot_vol_smile, plot_model_errors, plot_hedge_errors, pricing_error_summary

set_project_style()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Ingestion

Deribit BTC/ETH options are quoted in coin units, so the preprocessing step converts each option quote into USD using the option's reported underlying price.

In [ ]:
client = DeribitClient(cache_dir=PROJECT_ROOT / "data" / "cache", ttl_seconds=300)
snapshot = fetch_market_snapshot(("BTC", "ETH"), client=client, use_cache=True)

option_frames = []
for currency, payload in snapshot.items():
    options = normalize_option_chain(payload["book"], payload["instruments"])
    options["currency"] = currency
    option_frames.append(filter_liquid_options(options))

options_all = combine_currency_frames(option_frames)
options_all[["currency", "instrument_name", "option_type", "strike", "underlying_price", "time_to_maturity", "market_price_usd", "mark_iv_decimal"]].head()

## 2. Historical Jump Initialization

Jump parameters are initialized from historical Deribit index returns. The robust jump filter avoids letting extreme shocks inflate the diffusive volatility estimate.

In [ ]:
historical_params = {}
return_frames = {}

for currency, payload in snapshot.items():
    params, returns = estimate_historical_merton_params(payload["index"], z_threshold=3.0)
    historical_params[currency] = params
    return_frames[currency] = returns
    print(currency, params)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for ax, currency in zip(axes, ["BTC", "ETH"]):
    plot_return_jumps(return_frames[currency], ax=ax, title=f"{currency} Robust Jump Detection")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "return_jump_detection.png", dpi=180)

## 3. Baseline Volatility Smiles

The raw Deribit mark-IV smile is the empirical target. The project question is whether a jump component explains more of this smile than a single-volatility Black-Scholes baseline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, currency in zip(axes, ["BTC", "ETH"]):
    plot_vol_smile(options_all[options_all["currency"] == currency], ax=ax, title=f"{currency} Deribit Mark-IV Smile")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "deribit_iv_smiles.png", dpi=180)

## 4. Merton Calibration and Pricing Error

The calibration minimizes weighted relative price error against liquid option marks. This preserves the course baseline while introducing a jump-risk surface.

In [ ]:
calibration_results = {}
priced_frames = []

for currency in ["BTC", "ETH"]:
    currency_options = options_all[options_all["currency"] == currency].copy()
    result = calibrate_merton_to_options(currency_options, historical_params[currency], max_options=250)
    calibration_results[currency] = result
    priced = compare_model_errors(currency_options, result.params, bs_sigma=historical_params[currency].sigma)
    priced_frames.append(priced)
    print(currency, result)

priced_all = combine_currency_frames(priced_frames)
priced_all.groupby("currency").apply(pricing_error_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, currency in zip(axes, ["BTC", "ETH"]):
    plot_model_errors(priced_all[priced_all["currency"] == currency], ax=ax, title=f"{currency} Pricing Error: BS vs Merton")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pricing_error_comparison.png", dpi=180)

## 5. Binomial Convergence Benchmark

This satisfies the course requirement to compare the CRR tree against Black-Scholes as the number of periods grows.

In [ ]:
btc_spot = float(options_all.loc[options_all["currency"] == "BTC", "underlying_price"].median())
btc_sigma = historical_params["BTC"].sigma
convergence = convergence_table(btc_spot, btc_spot, 30 / 365.25, 0.01, btc_sigma, "call")
convergence

## 6. Greeks and Jump-Hedging Risk

The risk-management section compares Black-Scholes Greeks with jump-aware hedging under simulated Merton paths. The hedge simulation is deliberately discrete, so jump gaps create residual P&L.

In [ ]:
for currency in ["BTC", "ETH"]:
    spot = float(options_all.loc[options_all["currency"] == currency, "underlying_price"].median())
    sigma = historical_params[currency].sigma
    greeks = bs_greeks(spot, spot, 30 / 365.25, 0.01, sigma, "put")
    print(currency, greeks)

In [ ]:
currency = "BTC"
spot = float(options_all.loc[options_all["currency"] == currency, "underlying_price"].median())
params = calibration_results[currency].params
hedges = simulate_jump_hedge_experiment(
    spot,
    K=0.95 * spot,
    T=30 / 365.25,
    r=0.01,
    sigma=params.sigma,
    jump_intensity=params.jump_intensity,
    jump_mean=params.jump_mean,
    jump_vol=params.jump_vol,
    steps=30,
    paths=200,
    option_type="put",
)
fig, ax = plt.subplots(figsize=(8, 5))
plot_hedge_errors(hedges, ax=ax, title="BTC 30-Day Put Hedge Error Under Jump Paths")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "jump_hedge_error_distribution.png", dpi=180)
hedges.groupby("model")[["pnl", "absolute_pnl", "turnover"]].agg(["mean", "median", "std"])

## Report Takeaways

- Black-Scholes and CRR establish the required arbitrage-pricing baseline.
- The Merton jump component gives an explicit explanation for crash/skew risk in BTC/ETH options.
- The key empirical comparison is whether Merton reduces pricing errors and hedge losses in short-dated, out-of-the-money options where jump risk is most visible.